# Neural Tube Defect NICU Admissions, Maternal Characteristics, Clinical Features, and Regression Results in Addis Ababa, Ethiopia, 2021–2023 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Croissant URL**: `https://sen.science/doi/10.71728/senscience.89k5-pq0j/fair2.json`

> Note: This dataset package includes rich dataset-level metadata, but the supplied Croissant description indicates that `recordSet` is empty. Therefore, this notebook is designed to:
> 1. load and inspect the metadata,
> 2. safely check for record sets,
> 3. conditionally extract tabular records if any record sets are available at runtime, and
> 4. perform metadata-driven exploratory analysis when record-level data are not exposed through Croissant record sets.

All references to dataset entities in this notebook use their `@id` values whenever applicable.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`, and inspect key dataset-level properties.

In [ ]:
import json
from collections.abc import Mapping

import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.89k5-pq0j/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print('Name:', getattr(metadata, 'name', 'N/A'))
print('Description:', getattr(metadata, 'description', 'N/A'))

In [ ]:
# Convert metadata to a JSON-serializable object for inspection.
# We access dataset.metadata as a single object and only convert it afterward.
try:
    metadata_json = metadata.to_json()
except Exception:
    metadata_json = json.loads(str(metadata)) if isinstance(str(metadata), str) else {}

type(metadata_json)

## 2. Data Overview
Review available dataset-level metadata, distributions, and record sets.

Because Croissant datasets can contain multiple entities, we first inspect what is present before attempting extraction.

In [ ]:
# Display a compact dataset summary from metadata JSON
summary_fields = [
    '@id', 'name', 'identifier', 'version', 'license', 'citeAs',
    'datePublished', 'spatialCoverage', 'temporalCoverage'
]

summary = {field: metadata_json.get(field) for field in summary_fields}
pd.DataFrame(summary.items(), columns=['property', 'value'])

In [ ]:
# Inspect distribution @id values
distribution_entries = metadata_json.get('distribution', [])
distribution_ids = [entry.get('@id', 'N/A') for entry in distribution_entries if isinstance(entry, Mapping)]

distribution_df = pd.DataFrame({
    'distribution_@id': distribution_ids
})

distribution_df

In [ ]:
# Inspect record sets using their @id values when present
record_sets_metadata = metadata_json.get('recordSet', [])

if record_sets_metadata:
    record_set_rows = []
    for record_set in record_sets_metadata:
        if isinstance(record_set, Mapping):
            record_set_rows.append({
                'record_set_@id': record_set.get('@id', 'N/A'),
                'name': record_set.get('name', 'N/A'),
                'description': record_set.get('description', 'N/A')
            })
    display(pd.DataFrame(record_set_rows))
else:
    print('No record sets were declared in metadata_json["recordSet"].')

### Inspecting records from a record set
The template below shows how to iterate through records using a record set `@id`.

In this dataset, the provided package indicates that no record sets are declared, so the example is guarded and will only run if record sets become available.

In [ ]:
# Example: iterate through records for the first available record set @id, if any exist
record_set_ids = [
    rs.get('@id') for rs in record_sets_metadata
    if isinstance(rs, Mapping) and rs.get('@id')
]

if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print('Inspecting records from record set @id:', sample_record_set_id)
    for i, record in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print('No record set @id is available to pass to dataset.records(record_set=...).')

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame.

This section is fully dynamic and uses record set `@id` values. If no record sets are present, it falls back to extracting useful dataset-level metadata tables for analysis.

In [ ]:
# Extract data from each record set using record set @id values
dataframes = {}

if record_set_ids:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Record set {record_set_id} -> shape {dataframes[record_set_id].shape}')

    example_record_set_id = record_set_ids[0]
    print('\nColumns for example record set:')
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print('No Croissant record sets available. Creating metadata-driven tables instead.')

    dataframes['keywords_table'] = pd.DataFrame({
        'keyword': metadata_json.get('keywords', [])
    })

    dataframes['biases_table'] = pd.DataFrame({
        'dataBiases': metadata_json.get('dataBiases', [])
    })

    dataframes['limitations_table'] = pd.DataFrame({
        'dataLimitations': metadata_json.get('dataLimitations', [])
    })

    dataframes['use_cases_table'] = pd.DataFrame({
        'dataUseCases': metadata_json.get('dataUseCases', [])
    })

    dataframes['distribution_table'] = distribution_df.copy()

    for table_id, df in dataframes.items():
        print(f'{table_id} -> shape {df.shape}')

    display(dataframes['keywords_table'].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps.

Since this dataset package may not expose row-level record sets, this notebook performs two EDA paths:

1. **Record-set EDA** when Croissant records are available.
2. **Metadata EDA** when only dataset-level descriptive content is available.

This keeps the workflow robust while still following the template's style.

In [ ]:
# Helper function: find candidate numeric columns in a dataframe
def get_numeric_columns(df):
    return df.select_dtypes(include='number').columns.tolist()

# Helper function: find candidate grouping columns in a dataframe
def get_group_columns(df):
    return [col for col in df.columns if df[col].dtype == 'object']

if record_set_ids:
    record_set_id = record_set_ids[0]
    analysis_df = dataframes[record_set_id].copy()
    numeric_columns = get_numeric_columns(analysis_df)
    group_columns = get_group_columns(analysis_df)

    print('Selected record set @id for EDA:', record_set_id)
    print('Numeric columns:', numeric_columns)
    print('Grouping columns:', group_columns)

    if numeric_columns:
        numeric_field = numeric_columns[0]
        threshold = analysis_df[numeric_field].median()
        filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()

        print(f'Filtered records with {numeric_field} > {threshold}:')
        display(filtered_df.head())

        std_value = filtered_df[numeric_field].std()
        if pd.notnull(std_value) and std_value != 0:
            filtered_df[f'{numeric_field}_normalized'] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
            print(f'Normalized {numeric_field} for filtered records:')
            display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
        else:
            print(f'Skipping normalization for {numeric_field} because standard deviation is 0 or undefined.')

        if group_columns:
            group_field = group_columns[0]
            grouped_df = filtered_df.groupby(group_field, dropna=False)[numeric_field].mean().reset_index()
            print(f'Grouped data by {group_field}:')
            display(grouped_df.head())
    else:
        print('No numeric columns found in the selected record set. Descriptive EDA only.')
        display(analysis_df.head())
else:
    print('Running metadata-level EDA because no record sets are available.')

    keyword_table_id = 'keywords_table'
    numeric_field = 'keyword_length'
    group_field = 'keyword_initial'

    keywords_df = dataframes[keyword_table_id].copy()
    keywords_df[numeric_field] = keywords_df['keyword'].astype(str).str.len()
    keywords_df[group_field] = keywords_df['keyword'].astype(str).str[0].str.upper()

    threshold = 10
    filtered_df = keywords_df[keywords_df[numeric_field] > threshold].copy()
    print(f'Filtered records with {numeric_field} > {threshold}:')
    display(filtered_df.head())

    std_value = filtered_df[numeric_field].std()
    if pd.notnull(std_value) and std_value != 0:
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
        print(f'Normalized {numeric_field} for filtered records:')
        display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    else:
        print(f'Skipping normalization for {numeric_field} because standard deviation is 0 or undefined.')

    if group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field, dropna=False)[numeric_field]
            .mean()
            .reset_index()
            .sort_values(numeric_field, ascending=False)
        )
        print(f'Grouped data by {group_field}:')
        display(grouped_df.head())

In [ ]:
# Additional metadata exploration: convert selected list-based fields into a compact summary table
list_fields = [
    'keywords',
    'dataBiases',
    'dataLimitations',
    'dataUseCases',
    'dataCollectionType',
    'personalSensitiveInformation'
]

list_summary_rows = []
for field_id in list_fields:
    values = metadata_json.get(field_id, [])
    if isinstance(values, list):
        list_summary_rows.append({
            'field_@id_or_name': field_id,
            'item_count': len(values),
            'sample_values': '; '.join(map(str, values[:3]))
        })

list_summary_df = pd.DataFrame(list_summary_rows)
list_summary_df

## 5. Visualization
Visualize distributions or relationships in the available data.

If record sets are available, the plots will use the first accessible record set. Otherwise, the notebook visualizes metadata-derived summaries.

In [ ]:
plt.figure(figsize=(10, 5))

if record_set_ids:
    record_set_id = record_set_ids[0]
    viz_df = dataframes[record_set_id].copy()
    numeric_columns = get_numeric_columns(viz_df)

    if numeric_columns:
        numeric_field = numeric_columns[0]
        sns.histplot(viz_df[numeric_field].dropna(), bins=20, kde=True)
        plt.title(f'Distribution of {numeric_field} in record set {record_set_id}')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
    else:
        plt.text(0.5, 0.5, 'No numeric columns available for plotting.', ha='center', va='center')
        plt.axis('off')
else:
    keyword_table_id = 'keywords_table'
    viz_df = dataframes[keyword_table_id].copy()
    viz_df['keyword_length'] = viz_df['keyword'].astype(str).str.len()
    sns.histplot(viz_df['keyword_length'], bins=10, kde=True)
    plt.title('Distribution of keyword lengths')
    plt.xlabel('Keyword length')
    plt.ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))

if record_set_ids:
    record_set_id = record_set_ids[0]
    viz_df = dataframes[record_set_id].copy()
    numeric_columns = get_numeric_columns(viz_df)
    group_columns = get_group_columns(viz_df)

    if numeric_columns and group_columns:
        numeric_field = numeric_columns[0]
        group_field = group_columns[0]
        plot_df = (
            viz_df.groupby(group_field, dropna=False)[numeric_field]
            .mean()
            .reset_index()
            .sort_values(numeric_field, ascending=False)
            .head(10)
        )
        sns.barplot(data=plot_df, x=group_field, y=numeric_field)
        plt.title(f'Average {numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
    else:
        plt.text(0.5, 0.5, 'Not enough numeric/categorical columns for grouped visualization.', ha='center', va='center')
        plt.axis('off')
else:
    grouped_keywords_df = (
        filtered_df.groupby('keyword_initial', dropna=False)['keyword_length']
        .mean()
        .reset_index()
        .sort_values('keyword_length', ascending=False)
    )
    sns.barplot(data=grouped_keywords_df, x='keyword_initial', y='keyword_length')
    plt.title('Average keyword length by initial letter')
    plt.xlabel('Keyword initial')
    plt.ylabel('Average keyword length')

plt.tight_layout()
plt.show()

In [ ]:
# Visualize counts of selected list-based metadata fields
plt.figure(figsize=(10, 5))
sns.barplot(data=list_summary_df, x='field_@id_or_name', y='item_count')
plt.title('Counts of selected metadata lists')
plt.xlabel('Metadata field')
plt.ylabel('Number of items')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Conclusion
This notebook demonstrated how to explore a Croissant-described dataset with `mlcroissant` while keeping references aligned to entity `@id` values whenever they are available.

### Key observations
- The dataset metadata loaded successfully from the Croissant URL.
- The package provides rich dataset-level documentation, including description, use cases, biases, limitations, coverage, and distribution references.
- In the supplied Croissant package, `recordSet` is empty, so row-level extraction through `dataset.records(record_set=...)` is not available unless the source is updated to expose record sets.
- To keep the workflow useful, the notebook performed fallback metadata-driven analysis and visualizations.

### Next steps
- If the Croissant resource is updated with record sets, fields, and columns, the dynamic extraction cells in this notebook will begin loading them automatically using their `@id` values.
- You can extend this notebook by resolving distribution `@id` entries to downloadable files and then joining those files back to Croissant metadata for richer tabular analysis.
- For modeling tasks, first confirm that row-level records are exposed or accessible through linked distributions.

This notebook can be saved directly as `notebook.ipynb` and used as a reusable exploration template for Croissant datasets.